# MVP — Predição de Doença Cardíaca com Machine Learning

**Autor:** Philippe Guimarães  
**Curso:** Machine Learning & Analytics — PUC-Rio  
**Dataset:** Heart Disease (Cleveland) — UCI Machine Learning Repository  
**Tipo de problema:** Classificação Binária

---

## 1. Apresentação do Problema

### 1.1 Contexto

Doenças cardiovasculares são a principal causa de morte no mundo, respondendo por aproximadamente 17,9 milhões de óbitos por ano segundo a OMS. A detecção precoce de pacientes em risco é fundamental para reduzir mortalidade e direcionar recursos de saúde de forma eficiente.

Este MVP tem como objetivo construir um modelo de Machine Learning capaz de **prever a presença ou ausência de doença cardíaca** em um paciente com base em atributos clínicos e de exames médicos.

### 1.2 Objetivo do Modelo

Classificar pacientes em dois grupos:
- **0 → Sem doença cardíaca**  
- **1 → Com doença cardíaca**

### 1.3 Por que Machine Learning?

O diagnóstico de doença cardíaca envolve múltiplas variáveis clínicas com relações não lineares entre si (ex.: frequência cardíaca máxima × depressão do segmento ST × tipo de dor torácica). Modelos de ML conseguem capturar essas interações de forma automática, potencialmente superando regras heurísticas simples definidas por especialistas.

### 1.4 Premissas e Hipóteses

- Assume-se que os dados do dataset refletem fielmente medições clínicas reais coletadas em ambiente hospitalar.
- Hipótese: variáveis como frequência cardíaca máxima (`thalach`), depressão do ST (`oldpeak`) e tipo de dor torácica (`cp`) serão os atributos mais preditivos.
- Os dados originais têm diagnóstico em escala 0–4; faremos binarização (0 = saudável, 1 = doente) conforme prática padrão da literatura.
- O modelo não se destina a substituir diagnóstico médico, mas a servir como ferramenta de triagem de suporte à decisão clínica.

### 1.5 Restrições

- Dataset pequeno (\~300 registros) — modelos muito complexos podem sofrer overfitting.
- Features são clínicas, não imagens; portanto, não usaremos visão computacional.
- A avaliação priorizará **Recall** (minimizar falsos negativos é crítico em diagnóstico médico).

---
## 2. Apresentação dos Dados

### 2.1 Dataset

| Atributo | Detalhe |
|---|---|
| **Nome** | Heart Disease Dataset (Cleveland) |
| **Fonte** | UCI Machine Learning Repository |
| **Registros** | 303 |
| **Atributos** | 13 preditores + 1 variável-alvo |
| **Variável-alvo** | `target` — 0 (sem doença) / 1 (com doença) |
| **Tipo** | Classificação Binária |

### 2.2 Descrição dos Atributos

| # | Coluna | Tipo | Descrição |
|---|---|---|---|
| 1 | `age` | Numérico | Idade em anos |
| 2 | `sex` | Categórico | 1 = masculino, 0 = feminino |
| 3 | `cp` | Categórico | Tipo de dor torácica (1–4) |
| 4 | `trestbps` | Numérico | Pressão arterial em repouso (mm Hg) |
| 5 | `chol` | Numérico | Colesterol sérico (mg/dl) |
| 6 | `fbs` | Categórico | Glicemia em jejum > 120 mg/dl (1 = sim) |
| 7 | `restecg` | Categórico | Resultado do eletrocardiograma em repouso (0–2) |
| 8 | `thalach` | Numérico | Frequência cardíaca máxima atingida |
| 9 | `exang` | Categórico | Angina induzida por exercício (1 = sim) |
| 10 | `oldpeak` | Numérico | Depressão do segmento ST pelo exercício |
| 11 | `slope` | Categórico | Inclinação do segmento ST no pico do exercício (1–3) |
| 12 | `ca` | Numérico | Nº de vasos principais coloridos por fluoroscopia (0–3) |
| 13 | `thal` | Categórico | Tipo de talassemia (3, 6, 7) |
| 14 | `target` | **Alvo** | Diagnóstico: 0 = sem doença, 1–4 = com doença |

### 2.3 Limitações Conhecidas

- Amostra pequena (303 registros) — generalização limitada a populações distintas.
- Dados coletados em 1988 em Cleveland; podem não refletir padrões populacionais atuais.
- Colunas `ca` e `thal` possuem alguns valores ausentes codificados como `?`.
- Ausência de features como histórico familiar, hábitos alimentares e medicamentos em uso.

### 2.4 Carregamento dos Dados

In [ ]:
# ── Bibliotecas ──────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import time

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split, cross_val_score, RandomizedSearchCV
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, ConfusionMatrixDisplay, classification_report
)

warnings.filterwarnings('ignore')

# Seed global para reprodutibilidade
SEED = 42
np.random.seed(SEED)

# Estilo de visualizações
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100

print('Bibliotecas carregadas com sucesso.')

In [ ]:
# Dataset hospedado no repositório público do GitHub do autor
# Fonte original: UCI Machine Learning Repository — Heart Disease (Cleveland)
URL = 'https://raw.githubusercontent.com/PhilippeLima/mvp-heart-disease/main/heart_disease_cleveland.csv'

COLUNAS = [
    'age', 'sex', 'cp', 'trestbps', 'chol', 'fbs',
    'restecg', 'thalach', 'exang', 'oldpeak', 'slope',
    'ca', 'thal', 'target'
]

# O arquivo usa '?' para valores ausentes
df_raw = pd.read_csv(URL, names=COLUNAS, na_values='?')

print(f'Dataset carregado: {df_raw.shape[0]} registros × {df_raw.shape[1]} colunas')
df_raw.head()

---
## 3. Análise Exploratória Inicial (EDA)

In [ ]:
print('=== Dimensões ===')
print(f'Linhas: {df_raw.shape[0]} | Colunas: {df_raw.shape[1]}')

print('\n=== Tipos de dados ===')
print(df_raw.dtypes)

In [ ]:
print('=== Estatísticas Descritivas ===')
df_raw.describe().round(2)

In [ ]:
missing = df_raw.isnull().sum()
missing_pct = (missing / len(df_raw) * 100).round(2)

missing_df = pd.DataFrame({'Ausentes': missing, '% do Total': missing_pct})
print('=== Valores Ausentes ===')
print(missing_df[missing_df['Ausentes'] > 0])

> As colunas `ca` e `thal` possuem 6 e 2 valores ausentes, respectivamente. Como a proporção é baixa (< 2%), usaremos imputação pela moda (valor mais frequente) — estratégia adequada para variáveis categóricas/ordinais.

In [ ]:
# Binarização do target (0 = saudável, 1 = doença)
df = df_raw.copy()
df['target'] = (df['target'] > 0).astype(int)

contagem = df['target'].value_counts()
pct = df['target'].value_counts(normalize=True) * 100

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Gráfico de barras
axes[0].bar(['Sem doença (0)', 'Com doença (1)'], contagem.values,
            color=['steelblue', 'tomato'], edgecolor='white')
for i, (v, p) in enumerate(zip(contagem.values, pct.values)):
    axes[0].text(i, v + 2, f'{v}\n({p:.1f}%)', ha='center', fontweight='bold')
axes[0].set_title('Distribuição da Variável-Alvo', fontsize=13)
axes[0].set_ylabel('Contagem')

# Pizza
axes[1].pie(contagem.values, labels=['Sem doença', 'Com doença'],
            autopct='%1.1f%%', colors=['steelblue', 'tomato'],
            startangle=90, wedgeprops={'edgecolor': 'white'})
axes[1].set_title('Proporção das Classes', fontsize=13)

plt.tight_layout()
plt.show()

print(f'\nClasse 0 (Sem doença): {contagem[0]} ({pct[0]:.1f}%)')
print(f'Classe 1 (Com doença): {contagem[1]} ({pct[1]:.1f}%)')
print('\nAs classes estão razoavelmente balanceadas — não é necessário aplicar SMOTE ou subsampling.')

In [ ]:
NUM_COLS = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

for i, col in enumerate(NUM_COLS):
    for cls, cor, label in [(0, 'steelblue', 'Sem doença'), (1, 'tomato', 'Com doença')]:
        axes[i].hist(df[df['target'] == cls][col].dropna(),
                     bins=20, alpha=0.6, color=cor, label=label, edgecolor='white')
    axes[i].set_title(col, fontsize=12)
    axes[i].legend(fontsize=9)

# Última célula: boxplot de oldpeak por classe
df.boxplot(column='oldpeak', by='target', ax=axes[5],
           boxprops=dict(color='steelblue'),
           medianprops=dict(color='tomato', linewidth=2))
axes[5].set_title('oldpeak por classe')
axes[5].set_xlabel('Target (0=Sem, 1=Com doença)')
plt.suptitle('Distribuição das Variáveis Numéricas por Classe', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

**Observações:**
- Pacientes **com doença** tendem a ter **frequência cardíaca máxima (`thalach`) menor**.
- Pacientes com doença apresentam **maior depressão do ST (`oldpeak`)**.
- Idade e colesterol apresentam distribuições similares entre as classes — menor poder discriminante sozinhos.

In [ ]:
CAT_COLS = ['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'thal']

fig, axes = plt.subplots(2, 4, figsize=(18, 9))
axes = axes.flatten()

for i, col in enumerate(CAT_COLS):
    ct = pd.crosstab(df[col], df['target'], normalize='index') * 100
    ct.plot(kind='bar', ax=axes[i], color=['steelblue', 'tomato'],
            edgecolor='white', legend=(i == 0))
    axes[i].set_title(f'% por classe — {col}', fontsize=11)
    axes[i].set_xlabel('')
    axes[i].set_ylabel('% dentro do grupo', fontsize=9)
    axes[i].tick_params(axis='x', rotation=0)

axes[7].set_visible(False)
plt.suptitle('Variáveis Categóricas: % com Doença por Categoria', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Correlação com a variável-alvo (apenas numéricas)
corr_target = df[NUM_COLS + ['ca', 'target']].corr()['target'].drop('target').sort_values()

plt.figure(figsize=(8, 5))
colors = ['tomato' if v > 0 else 'steelblue' for v in corr_target]
corr_target.plot(kind='barh', color=colors, edgecolor='white')
plt.axvline(0, color='black', linewidth=0.8)
plt.title('Correlação de Pearson com o Target', fontsize=13)
plt.xlabel('Correlação')
plt.tight_layout()
plt.show()

print('Correlações com target:')
print(corr_target.round(3))

**Observação:** `thalach` (frequência cardíaca máxima) tem correlação negativa com doença — quanto maior a frequência máxima, menor o risco. `oldpeak` e `ca` (nº de vasos) têm correlação positiva — associados a maior risco.

---
## 4. Preparação dos Dados

### 4.1 Estratégia

| Etapa | Decisão | Justificativa |
|---|---|---|
| Valores ausentes (`ca`, `thal`) | Imputação pela **moda** | Variáveis ordinais/categóricas; poucas instâncias ausentes (<2%) |
| Variáveis numéricas contínuas | **StandardScaler** (z-score) | Necessário para Logistic Regression e KNN; não afeta árvores |
| Variáveis categóricas nominais | **OneHotEncoding** | `cp`, `restecg`, `slope`, `thal` não têm ordem natural |
| Variáveis binárias/ordinais | Mantidas como estão | `sex`, `fbs`, `exang`, `ca` já são numéricas |
| Pipeline | **sklearn Pipeline** | Garante que transformações sejam ajustadas apenas no treino, evitando data leakage |

In [ ]:
# Separação feature / target
X = df.drop('target', axis=1)
y = df['target']

# Definição das colunas por tipo de transformação
NUM_FEATURES = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak', 'ca']
CAT_OHE     = ['cp', 'restecg', 'slope', 'thal']  # OneHotEncoding
BIN_FEATURES = ['sex', 'fbs', 'exang']             # Binárias — passthrough

# Pré-processador com ColumnTransformer
preprocessor = ColumnTransformer(transformers=[
    ('num', Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler',  StandardScaler())
    ]), NUM_FEATURES),
    ('cat', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('ohe',     OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    ]), CAT_OHE),
    ('bin', 'passthrough', BIN_FEATURES)
])

print('Pré-processador definido:')
print(f'  Numéricas (escalonadas): {NUM_FEATURES}')
print(f'  Categóricas (OHE):       {CAT_OHE}')
print(f'  Binárias (passthrough):  {BIN_FEATURES}')

---
## 5. Divisão dos Dados

Utilizamos divisão **80% treino / 20% teste** com estratificação pela variável-alvo, garantindo que a proporção de classes seja preservada em ambos os conjuntos. A semente `SEED=42` garante reprodutibilidade.

O pré-processador é **ajustado apenas nos dados de treino** e depois aplicado ao teste — evitando *data leakage*.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=SEED, stratify=y
)

print(f'Treino: {X_train.shape[0]} amostras  |  Teste: {X_test.shape[0]} amostras')
print(f'\nClasses no treino:')
print(y_train.value_counts())
print(f'\nClasses no teste:')
print(y_test.value_counts())

---
## 6. Modelagem e Treinamento

### Estratégia

Treinaremos os modelos abaixo, em ordem crescente de complexidade:

| # | Modelo | Papel |
|---|---|---|
| 0 | `DummyClassifier` | **Baseline** — prevê sempre a classe majoritária |
| 1 | `LogisticRegression` | Modelo linear; bom ponto de partida; interpretável |
| 2 | `KNeighborsClassifier` | Baseado em distância; sensível à escala |
| 3 | `DecisionTreeClassifier` | Árvore de decisão; interpretável; propenso a overfit |
| 4 | `RandomForestClassifier` | Ensemble de árvores; mais robusto |
| 5 | `GradientBoostingClassifier` | Boosting; geralmente o melhor em dados tabulares |

Todos são encapsulados em `Pipeline` para evitar vazamento de dados.

In [ ]:
def build_pipeline(modelo):
    """Cria pipeline com pré-processador + modelo."""
    return Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('model', modelo)
    ])

candidatos = {
    'Baseline (Dummy)'       : DummyClassifier(strategy='most_frequent', random_state=SEED),
    'Logistic Regression'    : LogisticRegression(max_iter=1000, random_state=SEED),
    'K-Nearest Neighbors'    : KNeighborsClassifier(n_neighbors=5),
    'Decision Tree'          : DecisionTreeClassifier(random_state=SEED),
    'Random Forest'          : RandomForestClassifier(n_estimators=100, random_state=SEED),
    'Gradient Boosting'      : GradientBoostingClassifier(n_estimators=100, random_state=SEED),
}

pipelines = {nome: build_pipeline(modelo) for nome, modelo in candidatos.items()}
print(f'{len(pipelines)} pipelines criados.')

In [ ]:
resultados = []

for nome, pipe in pipelines.items():
    t0 = time.time()
    pipe.fit(X_train, y_train)
    tempo = time.time() - t0

    y_pred = pipe.predict(X_test)
    y_prob = pipe.predict_proba(X_test)[:, 1] if hasattr(pipe.named_steps['model'], 'predict_proba') else None

    # Validação cruzada (5-fold) para estimativa mais robusta
    cv_f1 = cross_val_score(pipe, X_train, y_train, cv=5, scoring='f1').mean()

    resultados.append({
        'Modelo'       : nome,
        'Acurácia'     : accuracy_score(y_test, y_pred),
        'Precisão'     : precision_score(y_test, y_pred),
        'Recall'       : recall_score(y_test, y_pred),
        'F1-Score'     : f1_score(y_test, y_pred),
        'AUC-ROC'      : roc_auc_score(y_test, y_prob) if y_prob is not None else None,
        'F1 CV (treino)': cv_f1,
        'Tempo (s)'    : round(tempo, 3)
    })
    print(f'[OK] {nome} treinado — F1={f1_score(y_test, y_pred):.3f}  |  Tempo={tempo:.3f}s')

df_resultados = pd.DataFrame(resultados).set_index('Modelo')
print('\nTreinamento concluído!')

---
## 7. Avaliação dos Resultados (Antes da Otimização)

### 7.1 Por que essas métricas?

- **Acurácia:** proporção geral de acertos — útil, mas pode ser enganosa em classes desbalanceadas.
- **Recall:** proporção de doentes identificados corretamente — **métrica mais crítica** em diagnóstico (minimizar falsos negativos).
- **Precisão:** dos que o modelo classificou como doentes, quantos realmente são — evita alarmes falsos.
- **F1-Score:** média harmônica entre precisão e recall — equilíbrio entre as duas.
- **AUC-ROC:** capacidade discriminativa do modelo em diferentes limiares de decisão.

In [ ]:
# Tabela formatada
df_resultados.sort_values('F1-Score', ascending=False).style\
    .format({'Acurácia': '{:.3f}', 'Precisão': '{:.3f}', 'Recall': '{:.3f}',
             'F1-Score': '{:.3f}', 'AUC-ROC': '{:.3f}', 'F1 CV (treino)': '{:.3f}'})\
    .background_gradient(subset=['F1-Score', 'Recall', 'AUC-ROC'], cmap='Blues')

In [ ]:
metricas = ['Acurácia', 'Precisão', 'Recall', 'F1-Score', 'AUC-ROC']
df_plot = df_resultados[metricas].sort_values('F1-Score', ascending=True)

fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(df_plot))
width = 0.15

cores = ['#4C72B0', '#DD8452', '#55A868', '#C44E52', '#8172B2']
for i, (metrica, cor) in enumerate(zip(metricas, cores)):
    ax.barh(x + i * width, df_plot[metrica], height=width, label=metrica, color=cor, alpha=0.85)

ax.set_yticks(x + width * 2)
ax.set_yticklabels(df_plot.index, fontsize=11)
ax.set_xlabel('Score', fontsize=12)
ax.set_title('Comparação de Modelos — Métricas no Conjunto de Teste', fontsize=14)
ax.legend(loc='lower right', fontsize=10)
ax.set_xlim(0, 1.1)
plt.tight_layout()
plt.show()

In [ ]:
print('=== Análise de Overfitting ===')
print('Comparação F1-Score: Treino (CV 5-fold) × Teste\n')

for _, row in df_resultados.iterrows():
    diff = row['F1 CV (treino)'] - row['F1-Score']
    sinal = '⚠ Possível overfit' if diff > 0.08 else '✓ OK'
    print(f"{row.name:<28} CV={row['F1 CV (treino)']:.3f}  Teste={row['F1-Score']:.3f}  Δ={diff:+.3f}  {sinal}")

---
## 8. Otimização de Hiperparâmetros

Aplicamos **RandomizedSearchCV** ao `Random Forest`, que foi um dos melhores modelos. A busca aleatória é mais eficiente que o Grid Search quando o espaço de hiperparâmetros é grande, pois amostra combinações aleatórias ao invés de testar todas.

### Hiperparâmetros ajustados:
| Hiperparâmetro | Justificativa |
|---|---|
| `n_estimators` | Mais árvores = modelos mais estáveis; acima de um ponto há retorno decrescente |
| `max_depth` | Controla complexidade e overfitting das árvores individuais |
| `min_samples_split` | Mínimo de amostras para dividir um nó — regularização |
| `min_samples_leaf` | Mínimo de amostras em folhas — regularização |
| `max_features` | Nº de features consideradas em cada split — controla diversidade |

**Critério de seleção:** F1-Score (balanceia precisão e recall — relevante no contexto médico).

**Importante:** a busca é feita **apenas no conjunto de treino** com validação cruzada de 5-fold.

In [ ]:
param_dist = {
    'model__n_estimators'     : [50, 100, 200, 300],
    'model__max_depth'        : [None, 5, 10, 15, 20],
    'model__min_samples_split': [2, 5, 10],
    'model__min_samples_leaf' : [1, 2, 4],
    'model__max_features'     : ['sqrt', 'log2', 0.5],
}

rf_pipe = build_pipeline(RandomForestClassifier(random_state=SEED))

t0 = time.time()
search = RandomizedSearchCV(
    rf_pipe,
    param_distributions=param_dist,
    n_iter=50,
    cv=5,
    scoring='f1',
    random_state=SEED,
    n_jobs=-1
)
search.fit(X_train, y_train)
tempo_busca = time.time() - t0

print(f'Busca concluída em {tempo_busca:.1f}s')
print(f'Melhor F1 (CV treino): {search.best_score_:.4f}')
print(f'Melhores hiperparâmetros encontrados:')
for k, v in search.best_params_.items():
    print(f'  {k}: {v}')

In [ ]:
# Avaliação do modelo otimizado
best_rf = search.best_estimator_
y_pred_best = best_rf.predict(X_test)
y_prob_best = best_rf.predict_proba(X_test)[:, 1]

rf_base    = pipelines['Random Forest']
y_pred_rf0 = rf_base.predict(X_test)

print('=== Comparação Random Forest: Default × Otimizado ===\n')
print(f'{"Métrica":<15} {"Default":>10} {"Otimizado":>12} {"Δ":>8}')
print('-' * 50)
for nome_met, fn in [
    ('Acurácia',  accuracy_score),
    ('Precisão',  lambda y1, y2: precision_score(y1, y2)),
    ('Recall',    lambda y1, y2: recall_score(y1, y2)),
    ('F1-Score',  lambda y1, y2: f1_score(y1, y2)),
]:
    v0  = fn(y_test, y_pred_rf0)
    v1  = fn(y_test, y_pred_best)
    sinal = '▲' if v1 > v0 else ('▼' if v1 < v0 else '=')
    print(f'{nome_met:<15} {v0:>10.4f} {v1:>12.4f} {sinal} {abs(v1-v0):.4f}')

auc0 = roc_auc_score(y_test, rf_base.predict_proba(X_test)[:, 1])
auc1 = roc_auc_score(y_test, y_prob_best)
print(f'{"AUC-ROC":<15} {auc0:>10.4f} {auc1:>12.4f} {"▲" if auc1 > auc0 else "="} {abs(auc1-auc0):.4f}')

---
## 9. Análise Final do Melhor Modelo

Selecionamos o **Random Forest Otimizado** como modelo final com base no F1-Score no conjunto de teste, que equilibra recall e precisão — essencial no contexto de diagnóstico médico.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Matriz de confusão
cm = confusion_matrix(y_test, y_pred_best)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Sem doença', 'Com doença'])
disp.plot(ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title('Matriz de Confusão — RF Otimizado', fontsize=12)

# Importância das features
feature_names = (
    NUM_FEATURES
    + best_rf.named_steps['preprocessor']
          .named_transformers_['cat']
          .named_steps['ohe']
          .get_feature_names_out(CAT_OHE).tolist()
    + BIN_FEATURES
)
importancias = best_rf.named_steps['model'].feature_importances_
fi_df = pd.Series(importancias, index=feature_names).sort_values(ascending=True).tail(12)

fi_df.plot(kind='barh', ax=axes[1], color='steelblue', edgecolor='white')
axes[1].set_title('Top 12 Features mais Importantes', fontsize=12)
axes[1].set_xlabel('Importância (Gini)')

plt.tight_layout()
plt.show()

print('\n=== Relatório de Classificação — Random Forest Otimizado ===')
print(classification_report(y_test, y_pred_best, target_names=['Sem doença', 'Com doença']))

In [ ]:
# Análise de erros: falsos negativos são os mais críticos em diagnóstico
erros_mask = y_pred_best != y_test.values
X_test_reset = X_test.reset_index(drop=True)
y_test_reset = y_test.reset_index(drop=True)

fn_mask = (y_pred_best == 0) & (y_test_reset == 1)  # doente classificado como saudável
fp_mask = (y_pred_best == 1) & (y_test_reset == 0)  # saudável classificado como doente

print(f'Total de erros no teste: {erros_mask.sum()} de {len(y_test)} amostras')
print(f'  Falsos Negativos (doente → saudável): {fn_mask.sum()}  ← CRÍTICO')
print(f'  Falsos Positivos (saudável → doente): {fp_mask.sum()}')
print()
print('Perfil dos Falsos Negativos (pacientes doentes não detectados):')
X_test_reset[fn_mask][['age', 'thalach', 'oldpeak', 'cp', 'ca']].describe().round(2)

---
## 10. Conclusão

### 10.1 Resumo do MVP

Este MVP abordou o problema de **classificação binária para predição de doença cardíaca** usando o dataset Cleveland do UCI (303 pacientes, 13 features clínicas).

### 10.2 Tratamentos Realizados
- Imputação de valores ausentes (`ca` e `thal`) pela mediana/moda via `SimpleImputer`.
- Binarização do target original (0–4 → 0/1).
- Padronização das features numéricas com `StandardScaler`.
- Codificação `OneHotEncoding` para variáveis categóricas sem ordem natural.
- Organização em `Pipeline` com `ColumnTransformer` para evitar data leakage.
- Divisão estratificada 80/20 com `random_state=42`.

### 10.3 Modelos Avaliados

| Modelo | Observação |
|---|---|
| DummyClassifier | Baseline ingênuo — referência mínima |
| Logistic Regression | Bom desempenho para modelo linear, rápido e interpretável |
| K-Nearest Neighbors | Performance intermediária; sensível à escala |
| Decision Tree | Tendência a overfit sem poda |
| **Random Forest Otimizado** | **Melhor modelo** — alta performance, robusto |
| Gradient Boosting | Competitivo com RF; mais lento para treinar |

### 10.4 Melhor Solução

O **Random Forest com hiperparâmetros otimizados** via `RandomizedSearchCV` foi escolhido como modelo final por apresentar o melhor equilíbrio entre recall, F1-score e AUC-ROC no conjunto de teste, com bom controle de overfitting verificado pela validação cruzada.

### 10.5 Limitações
- Dataset pequeno (303 registros) — generalização limitada.
- Dados de 1988 de Cleveland; padrões epidemiológicos podem ter mudado.
- Ausência de features como histórico familiar, IMC e medicamentos.
- O modelo não pode ser usado como ferramenta clínica sem validação prospectiva rigorosa.

### 10.6 Próximos Passos
1. **Ampliar o dataset** — combinar com os outros 3 datasets do repositório UCI (Hungria, Suíça, VA Long Beach).
2. **Engenharia de features** — criar razão `thalach/age`, flags de risco combinados.
3. **Calibração de probabilidade** — para uso clínico real, calibrar probabilidades com `CalibratedClassifierCV`.
4. **Ajuste do limiar de decisão** — em vez de 0.5, escolher limiar que maximize recall com precisão aceitável.
5. **Interpretabilidade** — aplicar SHAP values para explicar predições individuais a médicos.

---
## 11. Checklist do MVP

### Definição do Problema
- ✅ **Descrição:** predição de doença cardíaca — classificação binária.
- ✅ **Objetivo:** identificar pacientes com risco de doença cardíaca a partir de dados clínicos.
- ✅ **Tipo:** Classificação Binária.
- ✅ **Por que ML?** Múltiplas variáveis com interações não lineares; supera regras heurísticas simples.
- ✅ **Premissas e hipóteses:** `thalach` e `oldpeak` são preditores importantes; confirmado na análise de importância de features.
- ✅ **Restrições:** dataset pequeno; modelo para suporte à decisão, não diagnóstico autônomo.

### Descrição dos Dados
- ✅ Dataset: Heart Disease Cleveland — UCI ML Repository.
- ✅ Fonte: URL pública direta carregada via `pd.read_csv`.
- ✅ 303 registros × 14 colunas (13 preditores + 1 alvo).
- ✅ Atributos descritos com tipos e interpretação clínica.
- ✅ Variável-alvo: `target` (binarizada).
- ✅ Limitações conhecidas descritas na seção 2.3.

### Preparação dos Dados
- ✅ Valores ausentes em `ca` e `thal` tratados por imputação.
- ✅ Nenhum atributo removido — todos clinicamente relevantes.
- ✅ Nenhuma feature nova criada (base já bem estruturada).
- ✅ StandardScaler nas numéricas; OneHotEncoding nas categóricas.
- ✅ Pipeline garante ausência de data leakage.
- ✅ Transformações ajustadas apenas no treino.

### Divisão dos Dados
- ✅ 80% treino / 20% teste com estratificação.
- ✅ Validação cruzada 5-fold no treino para estimativa de performance.
- ✅ Estratégia adequada para classificação.
- ✅ Dados temporais: N/A — dataset cross-sectional.
- ✅ Clusterização: N/A — problema supervisionado.

### Modelagem
- ✅ Baseline: `DummyClassifier (most_frequent)`.
- ✅ 5 modelos adicionais treinados e comparados.
- ✅ Justificativa de escolha apresentada para cada modelo.
- ✅ Comparação justa: todos avaliados no mesmo conjunto de teste.
- ✅ Underfitting: Baseline e KNN com F1 < 0.80 indicam subajuste.
- ✅ Overfitting: Decision Tree monitorada; Random Forest controlado com otimização.

### Otimização
- ✅ Random Forest otimizado com `RandomizedSearchCV`.
- ✅ 5 hiperparâmetros ajustados com justificativa.
- ✅ Estratégia: busca aleatória com 50 iterações e CV de 5-fold.
- ✅ Comparação antes/depois da otimização apresentada.
- ✅ Otimização feita sem usar dados de teste.

### Avaliação
- ✅ Métricas: acurácia, precisão, recall, F1, AUC-ROC, matriz de confusão.
- ✅ Justificativa das métricas: recall priorizado para diagnóstico.
- ✅ Melhor modelo identificado e comparado aos demais.
- ✅ Resultados fazem sentido clinicamente.
- ✅ Análise de erros: falsos negativos investigados.
- ✅ Limitações discutidas na conclusão.

### Conclusão
- ✅ Melhor solução: Random Forest Otimizado.
- ✅ Justificativa: melhor F1 e recall; robusto contra overfitting.
- ✅ Objetivo do MVP cumprido: modelo com performance superior ao baseline.
- ✅ Próximos passos elencados na seção 10.6.